In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import math
import urllib.request


print("using device is idk wait i found it still loading ")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"your using :{device} ")
#we have to download shakespare dataset from internet


# ════════════════════════════════
# 1. DOWNLOAD SHAKESPEARE
# ════════════════════════════════


url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'shakespeare.txt')
with open('shakespeare.txt', 'r') as f:
    text = f.read()
print(f"Total characters :{len(text):,}")
print(f"sample: \n{text[:200]}")


# now we have convert into charater to function and learnthe model 
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ . CHARACTER LEVEL TOKENIZER ══════════════════════════════════════════════════════════
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"vocab size : {vocab_size}")
print(f"characters : {''.join(chars)}")
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode("hello"))
print(decode(encode('hello')))
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 3. TRAIN/VAL SPLIT ═══════════════════════════════════════════════════════════════════
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[:n]
print(f"train : {len(train_data):,} val: {len(val_data):,}")

# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 4. BATCH GENERATOR ═══════════════════════════════════════════════════════════════════
def get_batch(split,batch_size=32,block_size = 128):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+i + block_size] for i in ix])
    return x.to(device), y.to(device)

#════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 4.  MODEL  ═══════════════════════════════════════════════════════════════════
def scaled_dot_product_attention(Q,K,V , mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,K.transpose(-2,-1))
    scores = scores / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weigths = F.softmax(scores,dim=-1)
    return torch.matmul(weigths,V), weigths


# head 
class CausalMultiheadattention(nn.Module):
    def __init__(self,d_model,num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads


using device is idk wait i found it still loading 
your using :cpu 
Total characters :1,115,394
sample: 
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you
vocab size : 65
characters : 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
[46, 43, 50, 50, 53]
hello
train : 1,003,854 val: 1,003,854
